# 02. Baselines

## Load the data and split

Show the size of the training set and the test set

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.dataset import load_clients, add_year_block, time_split

df = add_year_block(load_clients())
train, test = time_split(df)
print(len(train), len(test))

32940 8236


## Feature list

Split the data into personal information about the client, information about the contact, and information about the economy.

One point of attention is that 'campaign' counts the number of calls to a client, including the one being predicted.

In [2]:
TARGET = "subscribed"

CLIENT  = ["age", "job", "marital", "education", "default", "housing", "loan"]
CONTACT = ["contact", "month", "day_of_week", "campaign", "pdays", "previous", "poutcome"]
MACRO   = ["emp_var_rate", "cons_price_idx", "cons_conf_idx", "euribor3m", "nr_employed"]

# Deliberately left out:
#   duration: only known after the call ends (leakage)
#   row_id, year_block, month_num: express time, not information about the client
FEATURES = CLIENT + CONTACT + MACRO

## Majority baseline model

Here we train a majority baseline model, which is the dumbest model that still counts as a model.
It ignores every feature and simply predicts whichever answer was most frequent in the training data.
In this case it will always predict 'no' for the subscription.
This is done initially because a 'smart' model should beat this: a 70% accuracy score, say, looks good on paper but is worthless if a dumb model like this one gets about the same.

The accuracy is mathematically defined as
$$\text{Accuracy} = \frac{\text{Correct classifications}}{\text{Total classifications}} = \frac{\mathrm{TP} + \mathrm{TN}}{\mathrm{TP} + \mathrm{TN} + \mathrm{FP} + \mathrm{FN}},$$
where $\mathrm{TP}$, $\mathrm{TN}$, $\mathrm{FP}$, and $\mathrm{FN}$ are the true positives, true negatives, false positives, and false negatives, respectively.
That is, the accuracy describes how many predictions were correct compared to how many were made.
A model that always gives the most common answer will score high in accuracy when the sample contains very few occurrences of the minority answer.

The precision is defined as
$$\text{Precision} = \frac{\mathrm{TP}}{\mathrm{TP} + \mathrm{FP}} .$$
This means that we compare the predicted number of positives that were actually correct to the total predicted positives.
The precision is meaningless in the majority baseline model because there are no predicted positives by construction, so in this case we will not evaluate it.

The recall is defined as
$$\text{Recall} = \frac{\mathrm{TP}}{\mathrm{TP} + \mathrm{FN}} .$$
This means that we compare the number of predicted positives that were correct to the true number of positives (i.e., those that were correctly predicted as positive plus those that were incorrectly predicted as negative).

Another metric is the $\text{ROC AUC}$ score.
To explain this we start with a model that gives each point a score, with a higher score meaning a higher chance of that point being predicted as positive.
Then we draw a line at a certain threshold and predict every point above the line as positive and every point below the line as negative.
For each value of the threshold we will have a certain number of $\mathrm{TP}$, $\mathrm{TN}$, $\mathrm{FP}$, and $\mathrm{FN}$, and can compute the precision and recall scores.
We can also calculate the false positive rate, namely
$$\text{FPR} = \frac{\text{FP}}{\text{FP} + \text{TN}},$$
which is the share of negative points that were wrongly flagged as positive.
If we plot the $\text{FPR}$ on the $x$ axis and the recall on the $y$ axis, we obtain the so-called $\text{ROC}$ curve.
The $\text{ROC AUC}$ score is the area under that curve.
It has a simple meaning: it's the probability that a randomly chosen positive gets a higher score than a randomly chosen negative.
Thus, a $\text{ROC AUC}$ score equal to $0.5$ means our model is the same as guessing randomly, and a $\text{ROC AUC}$ score of less than $0.5$ means the model is worse than a random guess.
In this case it's better to flip the prediction.

One last metric that deserves attention is the $\text{PR AUC}$ score.
For this we plot the precision on the $y$ axis as a function of the recall on the $x$ axis, again by varying the threshold.
In other words, we look at how much precision we lose as we increase the recall.
The curve is expected to be monotonically decreasing: intuitively, as we increase the recall of the model (correctly flag more of the correct points) we  will also lower its precision (because generally we will also flag more points that are actually negative).
The $\text{PR}$ score is the area under this curve and is a measure of the average precision of the model, computed by weighting by how much recall changing the threshold adds.

In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, average_precision_score

# Capital X, low y is a long-standing ML convention
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)  # In this case we never actually look at X_train (features ignored)

pred = dummy.predict(X_test)  # Returns 0 or 1 per row, in this case all rows will be 0
scores = dummy.predict_proba(X_test)[:, -1]  # predict_proba returns a column with P(class 0) and P(class 1), keep only the former

print('Accuracy: ', accuracy_score(y_test, pred))  # Share of all predictions that were correct (can be high with a 'dumb' model)
print('Recall:', recall_score(y_test, pred))  # Share of actual subscribers that the model found (will be 0 here)
print('ROC AUC:', roc_auc_score(y_test, pred))  # Chance that a random subscriber gets a higher score than a random non-subscriber: 0.5 means no ranking skill
print('PR AUC:', average_precision_score(y_test, pred))  # Precision vs recall

Accuracy:  0.6917192812044682
Recall: 0.0
ROC AUC: 0.5
PR AUC: 0.3082807187955318


We can explain these results quite easily:
- The accuracy is simply the rate of non-subscribers
- We didn't predict a single subscription, so the recall is zero
- Every score is tied and there's no ranking, so $\text{ROC AUC}$ is $0.5$
- $\text{PR AUC}$ is simply the subscription rate for the test set. This is because in our dummy model, all points get either flagged as positive or not at the same time. Precision goes up from 0 to the subscriber rate in a single step, and thus the area under the curve is $1 \times 0.308 = 0.308$.

To understand the last explanation we note that the area under the curve is computed by joining the points as a straight line in scikit's `average_precision_score`.

We also check the accuracy score for each year separately:

In [4]:
for block, part in test.groupby('year_block'):
    print(block, accuracy_score(part[TARGET], dummy.predict(part[FEATURES])))

1 0.7627063774684364
2 0.47862001943634597


## Logistic regression

### Preparation and preprocessing

We first slightly modify the dataset. Currently it contains a column `pdays` that represents the number of days since a client was contacted for the last time. When this was never the case, it has the value 999, which is a huge number. To amend this we create an extra column with 0 or 1 depending on whether the client was contacted before, and replace the 999 by 0.

In [5]:
def prepare(df):
    X = df[FEATURES].copy()
    X['contacted_before'] = (df['pdays'] != 999).astype(int)
    X['pdays'] = X['pdays'].where(X['pdays'] != 999, 0)
    return X

X_train, X_test = prepare(X_train), prepare(X_test)

We further separate the columns into categorical and numeric.
The categorical data is transformed into individual 0/1 columns for each item in a category (i.e., 'job' becomes 'job_admin', 'job_services', etc.), and `handle_unknown='ignore'` ensures that unknown data is set to 0 instead of raising an error.
Finally, `StandardScaler` scales the numerical data to have average 0 and spread 1, which helps the regression.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
numeric = [c for c in X_train.columns if c not in categorical]

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
    ('num', StandardScaler(), numeric)
])

### Full regression

We make a logistic regression which takes all the features into account and assigns each point a probability between 0 and 1.

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logreg = Pipeline([
    ('prep', preprocess),
    ('model', LogisticRegression(max_iter=10000))
])
logreg.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['age','job','marital',...,'euribor3m','nr_employed','contacted_before']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaini

We now display the $\text{ROC AUC}$ and $\text{PR AUC}$ scores of the test data by using the logistic regression made on the training data.
We do this for the full test data as well as resolving by year (tail of year 1 and full year 2 data).

In [8]:
def report(y, scores):
    return {'ROC_AUC': roc_auc_score(y, scores),
            'PR_AUC': average_precision_score(y, scores),
            'base_rate': y.mean()}

eval_sets = {'full_test': test,
         'block_1_tail': test[test['year_block']==1],  # This corresponds only to the tail of block 1
         'block_2': test[test['year_block']==2]}

for name, subset in eval_sets.items():
    s = logreg.predict_proba(prepare(subset))[:, 1]
    print(f'{name}:', report(subset[TARGET], s))

full_test: {'ROC_AUC': 0.747901872443385, 'PR_AUC': 0.5270723698616026, 'base_rate': np.float64(0.3082807187955318)}
block_1_tail: {'ROC_AUC': 0.732940713906846, 'PR_AUC': 0.4394609826469711, 'base_rate': np.float64(0.2372936225315636)}
block_2: {'ROC_AUC': 0.6544178521248363, 'PR_AUC': 0.6317120361007937, 'base_rate': np.float64(0.5213799805636541)}


The $\text{ROC AUC}$ score for each block is larger than $0.5$, and every $\text{PR AUC}$ score is above the base rate.
This means that the model does have real ranking skill.

The skill drops in the new year. In this case the $\text{PR AUC}$ score is only about $1.2\times$ the base rate, whereas it is $1.9\times$ in the tail of year 1.

Interestingly, the full $\text{ROC AUC}$ score is higher than both individual ones.
This likely means that the economic data is significantly influencing the fit with information that has nothing to do with the clients themselves.
If this is the case, the year-resolved numbers are more honest than the full test scores.

Let us also compute the average probability that a subscriber gets predicted.

In [9]:
for name, subset in eval_sets.items():
    s = logreg.predict_proba(prepare(subset))[:, 1]
    print(name, 'mean_predicted:', s.mean().round(3), 'actual:', subset[TARGET].mean().round(3))

full_test mean_predicted: 0.254 actual: 0.308
block_1_tail mean_predicted: 0.204 actual: 0.237
block_2 mean_predicted: 0.407 actual: 0.521


The model underpredicts in all cases.
However, it seems to perform well in the sense that the subscription rate in the training data is $6.4\%$.
In other words, even though the model was trained under a low subscription rate, it predicts a significantly higher number of subscribers once applied to the test data.

### Regression without macro

To test our hypothesis that the model is being overly influenced by the macroeconomic data, let us ignore this part of the data in a new regression

In [10]:
def make_logreg(categorical, numeric):
    prep = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
        ('num', StandardScaler(), numeric)
    ])
    return Pipeline([
        ('prep', prep),
        ('model', LogisticRegression(max_iter=1000))
    ])

numeric_no_macro = [c for c in numeric if c not in MACRO]
logreg_no_macro = make_logreg(categorical, numeric_no_macro)
logreg_no_macro.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['age','job','marital',...,'euribor3m','nr_employed','contacted_before']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaini

In [11]:
def evaluate(model, label):
    for name, subset in eval_sets.items():
        s = model.predict_proba(prepare(subset))[:, 1]
        r = report(subset[TARGET], s)
        print(f"{label:10s} {name:10s}: ROC {r['ROC_AUC']:.3f}, PR {r['PR_AUC']:.3f}, base {r['base_rate']:.3f}, mean_pred {s.mean():.3f}")

evaluate(logreg, "with macro")
evaluate(logreg_no_macro, "no macro")

with macro full_test : ROC 0.748, PR 0.527, base 0.308, mean_pred 0.254
with macro block_1_tail: ROC 0.733, PR 0.439, base 0.237, mean_pred 0.204
with macro block_2   : ROC 0.654, PR 0.632, base 0.521, mean_pred 0.407
no macro   full_test : ROC 0.700, PR 0.496, base 0.308, mean_pred 0.175
no macro   block_1_tail: ROC 0.672, PR 0.400, base 0.237, mean_pred 0.140
no macro   block_2   : ROC 0.645, PR 0.625, base 0.521, mean_pred 0.279


Indeed, removing the macro data has made the model predictions worse.
We see that macro data helps the model greatly in the tail of block 1 (where removing it has made the $\text{ROC AUC}$ score drop from $0.733$ to $0.672$), whereas for block 2 it hasn't really changed much (a drop of $0.009$ is most likely just noise).

What this means is that macro data might act as a "clock", where predictions depend not so much on the client that is being called but more on the moment of the call.
We test this by checking the predictions of may only.

In [12]:
may = test[(test['year_block'] == 1) & (test['month'] == 'may')]
print('n_clients:', len(may), ', no. distinct euribors:', may['euribor3m'].nunique(), ', subscription rate:', may[TARGET].mean().round(3))

for mdl, label in [(logreg, 'with macro'), (logreg_no_macro, 'no macro')]:
    s = mdl.predict_proba(prepare(may))[:, 1]
    print(label, round(roc_auc_score(may[TARGET], s), 3))

n_clients: 3274 , no. distinct euribors: 9 , subscription rate: 0.101
with macro 0.584
no macro 0.592


Removing the macro data from a single months almost hasn't changed the results.
This confirms our hypothesis: over a single month the macro data varies very little, and so does the prediction.
The two seem to be significantly correlated.
For now we will simply keep this in mind and carry on.